# Figure 2 — LLM extraction of prostate subtypes + platinum enrichment

This notebook builds the grant's Figure 2, which makes two points:

1. **We can extract prostate-cancer subtype information from clinical notes using an
   LLM**, validated against the Baca lab's manual chart-review annotations
   (Panel A — NEPC-vs-rest classifier performance).
2. **The platinum-treated population is enriched for aggressive variants
   (AVPC/NEPC)** relative to the platinum-untreated population (Panel B — descriptive
   subtype landscape; Panel C — the formal enrichment statistic).

**Data note:** all paths below point at the cluster mount
(`/data/gusev/USERS/jpconnor/...`) and this notebook is meant to be run there, not
locally.

**Label-swap note:** an earlier version of this notebook had the platinum+/platinum-
labels swapped when building `label_distributions` (Panel B) — the `is_platinum ==
True` counts were mislabeled `'negative'` and vice versa, which silently reversed the
enrichment story. This version fixes that; see Panel B for the correct mapping and the
sanity check that confirms it.

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import fisher_exact, norm
from sklearn.metrics import confusion_matrix

from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, Patch

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 220,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

# Colorblind-safe categorical palette (fixed assignment, never cycled):
# slot 1 = blue -> platinum+, slot 8 = orange -> platinum-. Kept consistent
# across Panels B and C so color identity means the same thing everywhere.
COLOR_PLATINUM_POS = "#2a78d6"   # blue
COLOR_PLATINUM_NEG = "#eb6834"   # orange
COLOR_NEUTRAL_INK = "#52514e"    # secondary ink, for annotations/text only

def binary_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    metrics = {
        "Accuracy":  (tp + tn) / (tp + tn + fp + fn),
        "Precision": tp / (tp + fp) if (tp + fp) > 0 else 0,
        "Recall":    tp / (tp + fn) if (tp + fn) > 0 else 0,
        "TPR":       tp / (tp + fn) if (tp + fn) > 0 else 0,
        "FPR":       fp / (fp + tn) if (fp + tn) > 0 else 0,
        "TNR":       tn / (tn + fp) if (tn + fp) > 0 else 0,
        "FNR":       fn / (fn + tp) if (fn + tp) > 0 else 0,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
    }

    return pd.Series(metrics)


def wilson_ci(successes, n, z=1.96):
    """Wilson score interval for a binomial proportion. Returns (phat, lo, hi)."""
    if n == 0:
        return (np.nan, np.nan, np.nan)
    phat = successes / n
    denom = 1 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    half = (z * np.sqrt((phat * (1 - phat) + z**2 / (4 * n)) / n)) / denom
    return phat, max(0.0, center - half), min(1.0, center + half)

## Load data

- `baca_lab_annotations.csv` — manual chart-review ground truth (Baca lab).
- `LLM_v3_labels.tsv` — LLM-extracted subtype labels (`primary_label` in
  `{conventional, avpc, nepc, biomarker}`).
- `platinum_MRN_list.csv` — MRNs of patients who received platinum chemotherapy.

In [ ]:
NEPC_PROJ_PATH = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS")

LLM_LABEL_PATH = NEPC_PROJ_PATH / 'LLM_NEPC_labels/'

OUT_DIR = Path("/data/gusev/USERS/jpconnor/figures/CAIA/COMPASS")
OUT_DIR.mkdir(parents=True, exist_ok=True)

manual_annotations = pd.read_csv(LLM_LABEL_PATH / 'baca_lab_annotations.csv')
nepc_annotations = pd.read_csv(LLM_LABEL_PATH / 'LLM_v3_labels.tsv', sep='\t')
platinum_mrns = pd.read_csv(NEPC_PROJ_PATH / 'mrn_lists/platinum_MRN_list.csv')

# Platinum membership via set lookup (was an O(n^2) dict comprehension keyed by
# `mrn in platinum_mrns['DFCI_MRN'].unique()` for every row -- .isin() against a
# set is both clearer and linear).
platinum_set = set(platinum_mrns['DFCI_MRN'].unique())
nepc_annotations['is_platinum'] = nepc_annotations['DFCI_MRN'].isin(platinum_set)

print(f"nepc_annotations: {len(nepc_annotations):,} rows "
      f"({nepc_annotations['is_platinum'].sum():,} platinum+, "
      f"{(~nepc_annotations['is_platinum']).sum():,} platinum-)")

## Panel A — LLM validation (NEPC-vs-rest classifier)

Ground truth is the Baca lab manual annotation
(`simplified_manual_platinum_reason ∈ {nepc, squamous_transformation}` → `manual_NEPC`);
prediction is the LLM label (`primary_label == 'nepc'` → `LLM_NEPC`). Both are computed
on `merged_results`, the manual↔LLM join on `DFCI_MRN`.

In [ ]:
merged_results = (manual_annotations.drop(columns=['pathology_details', 'manual_platinum_reason'])
                  .merge(nepc_annotations.drop(columns=['has_nepc', 'has_avpc', 'has_molecular_avpc',
                                                        'avpc_criteria', 'visceral_met_pattern', 'num_snippets']),
                                               on='DFCI_MRN'))
merged_results['manual_NEPC'] = (merged_results['simplified_manual_platinum_reason'].isin(['nepc', 'squamous_transformation']))
merged_results['LLM_NEPC'] = merged_results['primary_label'] == 'nepc'

print(f"merged_results: {len(merged_results):,} rows, "
      f"{merged_results['manual_NEPC'].sum():,} manual-NEPC positive")

In [ ]:
# manual = truth, LLM = pred (correct argument order for binary_metrics).
metrics = binary_metrics(merged_results['manual_NEPC'], merged_results['LLM_NEPC'])

# Sanity check: confusion-matrix counts must reconstruct N and the manual-NEPC total.
assert metrics[["TP", "FP", "TN", "FN"]].sum() == len(merged_results), \
    "confusion matrix counts do not sum to len(merged_results)"
assert metrics["TP"] + metrics["FN"] == merged_results['manual_NEPC'].sum(), \
    "TP+FN does not match manual_NEPC positive count"

metrics

In [ ]:
def render_confusion_panel(ax, metrics):
    """Annotated 2x2 confusion matrix, LLM (rows) vs manual truth (cols)."""
    cm = np.array([[metrics["TN"], metrics["FP"]],
                   [metrics["FN"], metrics["TP"]]])
    im = ax.imshow(cm, cmap="Blues", vmin=0)
    labels = ["Non-NEPC", "NEPC"]
    ax.set_xticks([0, 1]); ax.set_xticklabels(labels)
    ax.set_yticks([0, 1]); ax.set_yticklabels(labels)
    ax.set_xlabel("Manual annotation (truth)")
    ax.set_ylabel("LLM label (prediction)")
    ax.set_title("Panel A1 — confusion matrix", fontsize=11, weight="bold")

    thresh = cm.max() / 2 if cm.max() > 0 else 0.5
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{int(cm[i, j]):,}", ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "#0b0b0b",
                    fontsize=13, weight="bold")
    for spine in ax.spines.values():
        spine.set_visible(False)
    return im


def render_metric_bar_panel(ax, metrics):
    """Compact metric bar: Accuracy, Precision, Recall, Specificity only
    (drops redundant TPR=Recall and FPR/FNR duplicates per the plan)."""
    metric_names = ["Accuracy", "Precision", "Recall", "Specificity"]
    # Specificity == TNR; reuse the existing TNR value under a clearer name.
    values = [metrics["Accuracy"], metrics["Precision"], metrics["Recall"], metrics["TNR"]]

    bars = ax.bar(metric_names, values, color=COLOR_PLATINUM_POS, width=0.6)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("Metric value")
    ax.set_title("Panel A2 — classifier metrics", fontsize=11, weight="bold")
    ax.tick_params(axis="x", rotation=0)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f"{v:.2f}",
                ha="center", va="bottom", fontsize=10, color="#0b0b0b")
    return bars


n_total = len(merged_results)
n_nepc_manual = int(merged_results['manual_NEPC'].sum())
caption_a = (f"N = {n_total:,} chart-reviewed patients; "
             f"{n_nepc_manual:,} manual-NEPC positive.")

# --- individual sub-panels (repo convention: always emit standalone sub-panels) ---
fig_a1, ax_a1 = plt.subplots(figsize=(4.2, 4.2))
render_confusion_panel(ax_a1, metrics)
fig_a1.suptitle(caption_a, fontsize=8.5, color=COLOR_NEUTRAL_INK, y=0.01)
fig_a1.tight_layout()
for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure2a1_confusion_matrix.{ext}"
    fig_a1.savefig(out)
    print(f"wrote {out}")
plt.show()

fig_a2, ax_a2 = plt.subplots(figsize=(5.0, 4.2))
render_metric_bar_panel(ax_a2, metrics)
fig_a2.suptitle(caption_a, fontsize=8.5, color=COLOR_NEUTRAL_INK, y=0.01)
fig_a2.tight_layout()
for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure2a2_metric_bar.{ext}"
    fig_a2.savefig(out)
    print(f"wrote {out}")
plt.show()

## Panel B — subtype landscape (descriptive, 4-class)

Fraction of each `primary_label` class within the platinum+ and platinum- groups
separately. **This is a descriptive landscape view, not the enrichment claim** — the
two groups are very different sizes (platinum+ n=200 vs platinum- n=1682) and the
platinum- group is the much larger, less richly annotated background population,
so within-group fractions here should not be read as an enrichment statistic. The
formal enrichment test (aggressive vs conventional, with an odds ratio and Fisher's
exact p-value) is Panel C.

**Label-swap fix:** `platinum_positive_labels` (built from `is_platinum == True`) is
assigned `platinum_status = 'positive'`, and `platinum_negative_labels`
(`is_platinum == False`) is assigned `'negative'`. An earlier version of this cell had
these two swapped, which silently flipped the legend. The assertion below cross-checks
against the known counts so this cannot silently regress again.

Expected counts: **platinum+** (n=200) = avpc 118 / nepc 36 / conventional 30 /
biomarker 16; **platinum-** (n=1682) = conventional 1090 / avpc 456 / biomarker 72 /
nepc 64.

In [ ]:
platinum_positive_labels = nepc_annotations.loc[nepc_annotations['is_platinum'], 'primary_label'].value_counts().reset_index()
platinum_negative_labels = nepc_annotations.loc[~nepc_annotations['is_platinum'], 'primary_label'].value_counts().reset_index()

platinum_positive_labels['frac'] = platinum_positive_labels['count'] / platinum_positive_labels['count'].sum()
platinum_positive_labels['platinum_status'] = 'positive'   # is_platinum == True  (n=200)

platinum_negative_labels['frac'] = platinum_negative_labels['count'] / platinum_negative_labels['count'].sum()
platinum_negative_labels['platinum_status'] = 'negative'    # is_platinum == False (n=1682)

label_distributions = pd.concat([platinum_positive_labels, platinum_negative_labels], ignore_index=True)

# --- Sanity check: guards against the swap regressing silently. ---
expected_pos_counts = {"avpc": 118, "nepc": 36, "conventional": 30, "biomarker": 16}
expected_neg_counts = {"conventional": 1090, "avpc": 456, "biomarker": 72, "nepc": 64}

pos_counts = platinum_positive_labels.set_index('primary_label')['count'].to_dict()
neg_counts = platinum_negative_labels.set_index('primary_label')['count'].to_dict()

assert pos_counts == expected_pos_counts, (
    f"platinum-positive counts do not match expected {expected_pos_counts}; "
    f"got {pos_counts}. The platinum_status labels may be swapped again."
)
assert neg_counts == expected_neg_counts, (
    f"platinum-negative counts do not match expected {expected_neg_counts}; "
    f"got {neg_counts}. The platinum_status labels may be swapped again."
)
assert platinum_positive_labels['count'].sum() == 200
assert platinum_negative_labels['count'].sum() == 1682

label_distributions

In [ ]:
# Fixed class order (not alphabetical / not count-sorted) so bar position is
# stable and groups the two aggressive classes together for readability.
CLASS_ORDER = ["conventional", "avpc", "nepc", "biomarker"]

fig_b, ax_b = plt.subplots(figsize=(7.5, 5))
sns.barplot(
    data=label_distributions,
    x='primary_label', y='frac', hue='platinum_status',
    order=CLASS_ORDER,
    hue_order=['positive', 'negative'],
    palette={'positive': COLOR_PLATINUM_POS, 'negative': COLOR_PLATINUM_NEG},
    ax=ax_b,
)
ax_b.set_xlabel("LLM primary label")
ax_b.set_ylabel("Fraction within platinum group")
ax_b.set_ylim(0, 1.0)
ax_b.set_title("Panel B — subtype landscape by platinum status (descriptive)",
               fontsize=11, weight="bold")

n_pos = int(platinum_positive_labels['count'].sum())
n_neg = int(platinum_negative_labels['count'].sum())
handles, _ = ax_b.get_legend_handles_labels()
ax_b.legend(handles, [f"Platinum+ (n={n_pos:,})", f"Platinum- (n={n_neg:,})"],
            title=None, loc="upper right")

caption_b = ("Fractions computed within each platinum group separately; groups differ "
             "greatly in size and are not a random sample of the same population — "
             "see Panel C for the base-rate-robust enrichment statistic.")
fig_b.text(0.5, -0.02, caption_b, ha="center", va="top", fontsize=8.5,
            color=COLOR_NEUTRAL_INK, wrap=True)
fig_b.tight_layout()
for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure2b_subtype_landscape.{ext}"
    fig_b.savefig(out)
    print(f"wrote {out}")
plt.show()

## Panel C — enrichment statistic (the claim)

Collapse `primary_label` to **aggressive** (avpc + nepc) vs **conventional**, excluding
`biomarker` (and any non-prostate-primary rows) from this contrast — stated explicitly
here rather than silently dropped. Test `P(platinum+ | aggressive)` vs
`P(platinum+ | conventional)` with a 2x2 odds ratio and Fisher's exact test, and plot
the two proportions with Wilson score confidence intervals.

Expected 2x2 (excluding biomarker): aggressive/platinum+ = 154, aggressive/platinum- =
520, conventional/platinum+ = 30, conventional/platinum- = 1090 → **OR ≈ 10.8**,
Fisher p ≪ 0.001, in the claimed direction (platinum+ enriched for aggressive variants).

In [ ]:
df = nepc_annotations.copy()

# Exclude 'biomarker' (and anything outside the 3 core classes) from the
# aggressive-vs-conventional contrast; stated here rather than silently dropped.
n_before = len(df)
df = df[df['primary_label'].isin(['conventional', 'avpc', 'nepc'])]
n_excluded = n_before - len(df)
print(f"Excluded {n_excluded:,} rows outside {{conventional, avpc, nepc}} "
      f"(e.g. 'biomarker') from the enrichment contrast; {len(df):,} rows remain.")

df['aggressive'] = df['primary_label'].isin(['avpc', 'nepc'])

# 2x2 table: rows = aggressive/conventional, cols = platinum True/False.
ct = pd.crosstab(df['aggressive'], df['is_platinum'])
ct = ct.reindex(index=[True, False], columns=[True, False])
ct.index = ['aggressive', 'conventional']
ct.columns = ['platinum+', 'platinum-']
print(ct)

# Sanity check against the plan's known margins.
assert ct.loc['aggressive', 'platinum+'] == 154
assert ct.loc['conventional', 'platinum+'] == 30
assert ct.loc['aggressive', 'platinum-'] == 520
assert ct.loc['conventional', 'platinum-'] == 1090

OR, p_value = fisher_exact(ct.values, alternative='greater')
print(f"\nOdds ratio (aggressive vs conventional, platinum+ vs platinum-): OR = {OR:.2f}")
print(f"Fisher's exact p-value (one-sided, OR > 1): p = {p_value:.3g}")

assert OR > 1, (
    "OR <= 1 -- the platinum_status labels are almost certainly swapped again; "
    "the expected direction is platinum+ enriched for aggressive variants (OR > 1)."
)

In [ ]:
# P(platinum+ | aggressive) vs P(platinum+ | conventional), with Wilson CIs.
n_aggressive = int(ct.loc['aggressive'].sum())
n_conventional = int(ct.loc['conventional'].sum())
platinum_given_aggressive = int(ct.loc['aggressive', 'platinum+'])
platinum_given_conventional = int(ct.loc['conventional', 'platinum+'])

p_agg, lo_agg, hi_agg = wilson_ci(platinum_given_aggressive, n_aggressive)
p_conv, lo_conv, hi_conv = wilson_ci(platinum_given_conventional, n_conventional)

print(f"P(platinum+ | aggressive)   = {platinum_given_aggressive}/{n_aggressive} = "
      f"{p_agg:.1%}  (95% CI {lo_agg:.1%}-{hi_agg:.1%})")
print(f"P(platinum+ | conventional) = {platinum_given_conventional}/{n_conventional} = "
      f"{p_conv:.1%}  (95% CI {lo_conv:.1%}-{hi_conv:.1%})")

In [ ]:
def render_enrichment_panel(ax):
    groups = ["Aggressive\n(AVPC + NEPC)", "Conventional"]
    props = [p_agg, p_conv]
    lo = [p_agg - lo_agg, p_conv - lo_conv]
    hi = [hi_agg - p_agg, hi_conv - p_conv]
    # Same two-slot palette as Panel B would use for platinum+/-, but here the
    # two bars are the *aggressive* vs *conventional* groups being compared on
    # the platinum+ rate -- use a single accent color since it's one series
    # (the y-value, P(platinum+ | group)), with group identity on the x-axis.
    bar_colors = [COLOR_PLATINUM_POS, "#9a9890"]  # accent vs. muted comparison

    bars = ax.bar(groups, props, yerr=[lo, hi], capsize=6,
                  color=bar_colors, width=0.55,
                  error_kw={"ecolor": COLOR_NEUTRAL_INK, "elinewidth": 1.5})
    ax.set_ylabel("P(platinum+ | subtype group)")
    ax.set_ylim(0, max(hi_agg, hi_conv) * 1.35)
    ax.set_title("Panel C — platinum enrichment among aggressive variants",
                 fontsize=11, weight="bold")

    for bar, v, n, k in zip(bars, props, [n_aggressive, n_conventional],
                             [platinum_given_aggressive, platinum_given_conventional]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.025,
                f"{v:.1%}\n({k}/{n})", ha="center", va="bottom", fontsize=9.5)

    ax.text(0.5, 0.92, f"OR = {OR:.1f}, Fisher's exact p = {p_value:.1e}",
            transform=ax.transAxes, ha="center", va="top",
            fontsize=10.5, weight="bold", color=COLOR_NEUTRAL_INK)
    return bars


fig_c, ax_c = plt.subplots(figsize=(6, 5.5))
render_enrichment_panel(ax_c)
caption_c = (f"Excludes 'biomarker' primary_label ({n_excluded:,} rows) from the "
             "aggressive/conventional contrast. Error bars are 95% Wilson score "
             "intervals. OR and p from Fisher's exact test (one-sided, OR > 1) on "
             "the 2x2 aggressive/conventional x platinum+/- table.")
fig_c.text(0.5, -0.05, caption_c, ha="center", va="top", fontsize=8.5,
            color=COLOR_NEUTRAL_INK, wrap=True)
fig_c.tight_layout()
for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure2c_enrichment.{ext}"
    fig_c.savefig(out)
    print(f"wrote {out}")
plt.show()

## Compiled Figure 2

Compose Panels A (confusion matrix + metric bar), B (subtype landscape), and C
(enrichment) into one figure using `plt.subplot_mosaic`, reusing the same render
functions as the individual sub-panels above so the compiled figure and the
standalone sub-panel files never drift apart.

In [ ]:
fig, axd = plt.subplot_mosaic(
    [["A1", "A2", "B"],
     ["C",  "C",  "B"]],
    figsize=(15, 9),
    width_ratios=[1, 1.1, 1.3],
    height_ratios=[1, 1],
)

render_confusion_panel(axd["A1"], metrics)
render_metric_bar_panel(axd["A2"], metrics)

sns.barplot(
    data=label_distributions,
    x='primary_label', y='frac', hue='platinum_status',
    order=CLASS_ORDER, hue_order=['positive', 'negative'],
    palette={'positive': COLOR_PLATINUM_POS, 'negative': COLOR_PLATINUM_NEG},
    ax=axd["B"],
)
axd["B"].set_xlabel("LLM primary label")
axd["B"].set_ylabel("Fraction within platinum group")
axd["B"].set_ylim(0, 1.0)
axd["B"].set_title("Panel B — subtype landscape (descriptive)", fontsize=11, weight="bold")
handles, _ = axd["B"].get_legend_handles_labels()
axd["B"].legend(handles, [f"Platinum+ (n={n_pos:,})", f"Platinum- (n={n_neg:,})"],
                title=None, loc="upper right", fontsize=9)

render_enrichment_panel(axd["C"])

fig.suptitle("Figure 2 — LLM-extracted prostate subtypes validate against manual "
             "annotation and reveal platinum enrichment for aggressive variants",
             fontsize=13, weight="bold", y=1.02)

full_caption = (
    f"(A) NEPC-vs-rest classifier, LLM vs Baca-lab manual annotation "
    f"(N={n_total:,}, {n_nepc_manual:,} manual-NEPC+). "
    f"(B) Descriptive subtype landscape, platinum+ (n={n_pos:,}) vs platinum- (n={n_neg:,}); "
    "not itself the enrichment claim. "
    f"(C) Platinum+ rate among aggressive (AVPC+NEPC) vs conventional patients "
    f"(biomarker excluded, {n_excluded:,} rows); Wilson 95% CIs; "
    f"OR = {OR:.1f}, Fisher's exact p = {p_value:.1e}."
)
fig.text(0.5, -0.03, full_caption, ha="center", va="top", fontsize=9,
          color=COLOR_NEUTRAL_INK, wrap=True)

fig.tight_layout()
for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure2_llm_subtype_platinum.{ext}"
    fig.savefig(out, bbox_inches="tight")
    print(f"wrote {out}")
plt.show()